# 02 Build long table timeseries
Convert MACHINE/RFM to long table: wafer_id, exp_id, label, t, var, value. Output: data/interim/timeseries_long.parquet

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "backend"))
from app.etchfdc.io.mat_reader import load_mat

DATA_RAW = ROOT / "data" / "raw"
DATA_INTERIM = ROOT / "data" / "interim"
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

In [ ]:
# Build long table from loaded .mat (adapt keys to your MACHINE/RFM structure)
rows = []
for fname in ["MACHINE_Data.mat", "RFM_DATA.mat"]:
    path = DATA_RAW / fname
    if not path.exists():
        continue
    data = load_mat(path)
    for key, arr in data.items():
        if hasattr(arr, 'shape') and arr.ndim >= 1:
            for t, val in enumerate(np.atleast_1d(arr).flat):
                rows.append({"wafer_id": "", "exp_id": key, "label": "", "t": t, "var": key, "value": float(val)})
if rows:
    long_df = pd.DataFrame(rows)
    long_df.to_parquet(DATA_INTERIM / "timeseries_long.parquet", index=False)
    print(long_df.head())
else:
    print("No data; add conversion logic for your .mat structure.")